# 04 · Cross-sell recommender (point of sale / checkout)

**Business problem.** About a third of orders contain a single item and the average basket is ~2.1 items.
When a cashier scans a boxer, what is the *one add-on most likely to be accepted that also earns good margin*?

**Method (right-sized for 139 products):**
* Build order × product baskets (retail POS + e-commerce; wholesale excluded because reseller bulk baskets behave differently).
* **Item→item confidence with shrinkage** `conf(i→j) = count(i&j) / (count(i) + k)` – rare items cannot produce spuriously high scores.
* Several items in the basket are combined with **noisy-or**: `1 − Π(1 − conf)`.
* **Margin-aware ranking**: `expected margin = probability × margin per unit`.
* **Stock mask** applied by the app (don't suggest what the branch does not have).

**Evaluation (offline, leave-one-out on future orders):** hide one product of each held-out multi-item order, recommend from the rest, and check whether the hidden product is in the top-*k*. Compared with the *most-popular-items* baseline.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                       # lets the notebook import the `cottonil` package
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cottonil import config as cfg
from cottonil.data_io import load_table, load_sales_lines, load_product_master, week_start, quick_profile
from cottonil.plotting import setup, NAVY, TEAL, ORANGE, RED, PALETTE
setup()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

DATA_DIR = cfg.find_data_dir()
SOURCE = cfg.data_source(DATA_DIR)
print(f"Data folder : {DATA_DIR}")
print(f"Data source : {SOURCE.upper()}" + ("   <-- synthetic test data; retrain on real data before trusting any number" if SOURCE == "synthetic" else ""))

Data folder : D:\HVIA Internship\task 2\cottonil-ai\COTTONIL_Data
Data source : REAL


In [2]:
from cottonil import crosssell as X
lines = load_sales_lines()
lines = lines[lines.sales_channel != "Wholesale / reseller"]
prods = load_table("products_catalog")
pid = prods.product_id.tolist(); pidx = {p: i for i, p in enumerate(pid)}
lines["pi"] = lines.product_id.map(pidx).astype(int)
last = lines.order_date.max()
print(f"{lines.sales_order_id.nunique():,} retail/e-com orders | {len(pid)} products | data ends {last.date()}")

485,831 retail/e-com orders | 139 products | data ends 2026-10-18


## 1 · Split by time: train → validation (6 months) → test (last 6 months)

In [3]:
test_start, val_start = last - pd.DateOffset(months=6), last - pd.DateOffset(months=12)
tr, va, te = (lines[lines.order_date < val_start], lines[(lines.order_date >= val_start) & (lines.order_date < test_start)], lines[lines.order_date >= test_start])
def baskets(df, n_min=1):
    B = X.basket_matrix(df.sales_order_id.values, df.pi.values, len(pid)); return B[np.asarray(B.sum(1)).ravel() >= n_min]
B_tr, B_va, B_te = baskets(tr), baskets(va, 2), baskets(te, 2)
print(f"train baskets {B_tr.shape[0]:,} | validation multi-item baskets {B_va.shape[0]:,} | test multi-item baskets {B_te.shape[0]:,}")
print(f"single-item share in train: {(np.asarray(B_tr.sum(1)).ravel() == 1).mean():.1%}")

train baskets 395,040 | validation multi-item baskets 31,477 | test multi-item baskets 29,922
single-item share in train: 32.4%


In [4]:
rng = np.random.default_rng(0)
def loo(B):
    """hide one random product per basket -> (seed matrix, hidden product index)"""
    B = B.tocsr(); seeds, hidden = B.toarray().astype(np.float32), np.zeros(B.shape[0], int)
    for r in range(B.shape[0]):
        cols = B.indices[B.indptr[r]:B.indptr[r + 1]]; h = rng.choice(cols); hidden[r] = h; seeds[r, h] = 0
    return seeds, hidden
def hit_mrr(P, hidden, seeds, ks=(1, 3, 5)):
    P = np.where(seeds > 0, -1, P)                                   # never recommend what is already in the basket
    rank = (P > P[np.arange(len(P)), hidden][:, None]).sum(1) + 1
    out = {f"hit@{k}": float((rank <= k).mean()) for k in ks}; out["MRR"] = float((1 / rank).mean()); return out

seeds_va, hid_va = loo(B_va)
rows = []
pop = np.asarray(B_tr.sum(0)).ravel() / B_tr.shape[0]
for k in (0, 5, 20, 50, 100):
    m = X.fit(B_tr, k_shrink=k); P = X.score_baskets(m["conf"], seeds_va)
    rows.append({"k_shrink": k, **hit_mrr(P, hid_va, seeds_va)})
tune = pd.DataFrame(rows).set_index("k_shrink"); tune.round(4)

,hit@1,hit@3,hit@5,MRR
k_shrink,,,,
0,0.0165,0.0469,0.0793,0.0718
5,0.0165,0.0469,0.0793,0.0718
20,0.0165,0.0469,0.0793,0.0718
50,0.0165,0.0468,0.0793,0.0718
100,0.0164,0.0468,0.0795,0.0718


In [5]:
K_BEST = int(tune["hit@3"].idxmax()); print("chosen k_shrink (best validation hit@3):", K_BEST)
# final evaluation on TEST. model is fitted on train + validation (all data before the test window)
B_fit = baskets(pd.concat([tr, va])); m_eval = X.fit(B_fit, k_shrink=K_BEST)
seeds_te, hid_te = loo(B_te)
P_model = X.score_baskets(m_eval["conf"], seeds_te)
pop_fit = np.asarray(B_fit.sum(0)).ravel(); P_pop = np.tile(pop_fit / pop_fit.sum(), (len(seeds_te), 1))
mpu_fit = (pd.concat([tr, va]).groupby("pi").margin_EGP.sum() / pd.concat([tr, va]).groupby("pi").quantity.sum()).reindex(range(len(pid))).fillna(0).values
P_marg = P_model * mpu_fit[None, :]
res = pd.DataFrame({"co-occurrence (by probability)": hit_mrr(P_model, hid_te, seeds_te), "co-occurrence (by expected margin)": hit_mrr(P_marg, hid_te, seeds_te),
                    "popularity baseline": hit_mrr(P_pop, hid_te, seeds_te)}).T
print(f"test: {len(seeds_te):,} held-out baskets, random-guess hit@5 would be ~{5 / len(pid):.1%}")
res.round(4)

chosen k_shrink (best validation hit@3): 0


test: 29,922 held-out baskets, random-guess hit@5 would be ~3.6%


,hit@1,hit@3,hit@5,MRR
co-occurrence (by probability),0.0154,0.0473,0.0794,0.0712
co-occurrence (by expected margin),0.0178,0.0469,0.0764,0.0700
popularity baseline,0.0152,0.0484,0.0816,0.0715


**Reading it.** *hit@3* = share of baskets where the hidden item appears in the top-3 suggestions.
The *margin-aware* ranker gives up a little hit-rate to push higher-margin items – that is the business trade-off the app exposes via a toggle.

In [6]:
# average margin per unit of what each ranker puts in its top-3 (when it is right or wrong)
def avg_margin_top3(P, seeds):
    P = np.where(seeds > 0, -1, P); top = np.argsort(-P, axis=1)[:, :3]; return float(mpu_fit[top].mean())
print("avg margin per unit in top-3 suggestions (EGP):",
      {"by probability": round(avg_margin_top3(P_model, seeds_te), 1), "by expected margin": round(avg_margin_top3(P_marg, seeds_te), 1), "popularity": round(avg_margin_top3(P_pop, seeds_te), 1)})

avg margin per unit in top-3 suggestions (EGP): {'by probability': 161.9, 'by expected margin': 282.9, 'popularity': 213.8}


## 2 · Fit the final model on ALL data and look at what it learned

In [7]:
B_all = baskets(lines); final = X.fit(B_all, k_shrink=K_BEST)
a = lines.groupby("pi").agg(q=("quantity", "sum"), m=("margin_EGP", "sum"), r=("net_line_EGP", "sum"))
recent = lines[lines.order_date > last - pd.Timedelta(days=365)].groupby("pi").agg(q=("quantity", "sum"), m=("margin_EGP", "sum"), r=("net_line_EGP", "sum"))
mpu = (recent.m / recent.q).reindex(range(len(pid))); mpu = mpu.fillna((a.m / a.q).reindex(range(len(pid)))).fillna(0).values
price = (recent.r / recent.q).reindex(range(len(pid))).fillna((a.r / a.q).reindex(range(len(pid)))).fillna(prods.listed_price_EGP).values
bundle = {**final, "product_ids": pid, "margin_per_unit": mpu.astype(np.float32), "price": price.astype(np.float32),
          "category": prods.category_en.tolist(), "name_ar": prods.product_name_ar.tolist(), "data_source": SOURCE}
cfg.save_model("crosssell", bundle)

# category-level lift (for the heatmap in the app and for business rules like 'Boxer + Socks bundle')
cats = prods.category_en.values; ucat = sorted(set(cats))
lines["ci"] = lines.category.astype(str).map({c: i for i, c in enumerate(ucat)}).astype(int)
Bc = X.basket_matrix(lines.sales_order_id.values, lines.ci.values, len(ucat)); mc = X.fit(Bc, k_shrink=0)
cl = pd.DataFrame(mc["lift"], index=ucat, columns=ucat).stack().rename("lift").reset_index(); cl.columns = ["seed_category", "add_on_category", "lift"]
cl["pair_orders"] = pd.DataFrame(mc["count"], index=ucat, columns=ucat).stack().values
cl["confidence"] = pd.DataFrame(mc["conf"], index=ucat, columns=ucat).stack().values
cfg.save_serving("category_lift", cl[cl.seed_category != cl.add_on_category].round(4))
top = cl[(cl.seed_category != cl.add_on_category) & (cl.pair_orders >= 500)].sort_values("lift", ascending=False).head(8)
top.round(3)

,seed_category,add_on_category,lift,pair_orders,confidence
5,Accessories,Kids underwear,0.801,1037.0,0.138
80,Kids underwear,Accessories,0.801,1037.0,0.012
27,Boys pajamas,Women bottoms,0.800,564.0,0.012
177,Women bottoms,Boys pajamas,0.800,564.0,0.076
131,Men tops,Girls underwear,0.800,2187.0,0.018
56,Girls underwear,Men tops,0.800,2187.0,0.199
138,Men tops,Socks,0.796,44150.0,0.366
168,Socks,Men tops,0.796,44150.0,0.198


In [8]:
# example: scan a Men boxer -> what would the till suggest?
box = prods[prods.category_en == "Men boxers"].product_id.iloc[0]
rec = X.recommend(bundle, [box], k=5)
rec["name"] = rec.product_id.map(dict(zip(pid, bundle["name_ar"]))); rec["category"] = rec.product_id.map(dict(zip(pid, bundle["category"])))
print("basket:", box, prods.set_index("product_id").loc[box, "product_name_ar"])
rec.round(3)

basket: PR0060 3 قطع بوكسر رجالى مارسريزيه


,product_id,prob,expected_margin,margin_per_unit,name,category
0,PR0090,0.023,12.138,524.773987,8 قطع بوكسر CN,Men boxers
1,PR0121,0.021,9.569,458.981995,تيشيرت مرسريزيه,Men tops
2,PR0081,0.024,9.483,401.415985,4 اندرشيرت كم استريتش,Men tops
3,PR0085,0.023,8.796,381.307007,5 قطع بوكسر كاجوال,Men boxers
4,PR0122,0.019,8.792,461.286011,بيجامة شتوي حريمي كود 2031,Women pajamas


In [9]:
cfg.save_metrics("04_crosssell", {"k_shrink": K_BEST, "n_test_baskets": int(len(seeds_te)), "test": res.round(4).to_dict("index"),
                                  "random_hit@5": 5 / len(pid), "single_item_share": float((np.asarray(B_all.sum(1)).ravel() == 1).mean()),
                                  "avg_margin_top3": {"probability": avg_margin_top3(P_model, seeds_te), "margin": avg_margin_top3(P_marg, seeds_te), "popularity": avg_margin_top3(P_pop, seeds_te)}}, DATA_DIR)

WindowsPath('D:/HVIA Internship/task 2/cottonil-ai/artifacts/metrics/04_crosssell.json')

## Takeaways
* If *co-occurrence* does not clearly beat *popularity* on real data, the dataset has no basket structure to exploit and a fixed "Boxer + Socks" bundle rule is as good as any model – the notebook tells you either way.
* The honest KPI to track after launch is **suggestion acceptance rate and basket margin** in an A/B test across branches, not offline hit-rate.